# Installation

Session options:
 * Accelerator: GPU T4 x2
 * Language: Python
 * Persistence: File Only
 * Environment: Pin to original environment


In [ ]:
from kaggle_secrets import UserSecretsClient
import subprocess
import os
import time

print("--- Step 1: Enforcing NVIDIA CUDA GPU Binaries ---")
# Overwrite the default CPU torch layout with the correct CUDA runtime wrapper
os.system("pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")

print("\n--- Step 2: Extracting Vault Credentials ---")
# Pull the value using the exact label name you verified
user_secrets = UserSecretsClient()
key = user_secrets.get_secret("TAILSCALE_KEY")

print("\n--- Step 3: Setting Up Clean Workspace ---")
os.system("cd /kaggle/working && rm -rf ComfyUI")
os.system("git clone https://github.com/comfyanonymous/ComfyUI.git")
os.system("cd /kaggle/working/ComfyUI && pip install -r requirements.txt")
os.system("curl -fsSL https://tailscale.com/install.sh | sh")

print("\n--- Step 4: Activating Isolated Proxy Network ---")
with open("tailscale.log", "w") as f:
    subprocess.Popen(
        ["tailscaled", "--tun=userspace-networking", "--socks5-server=localhost:1055"],
        stdout=f, stderr=f
    )
time.sleep(4)

# Authenticating the mesh node cleanly past the kernel watchdog
subprocess.run(["tailscale", "up", f"--authkey={key}", "--accept-dns=false"])

print("\n--- TUNNEL LIVE ---")
print("Verified Cloud Node IP Address:")
os.system("tailscale ip -4")

print("\n--- Step 5: Launching ComfyUI Server ---")
# Booting with explicit local binding to process traffic streaming through the proxy
os.system("cd /kaggle/working/ComfyUI && python main.py --listen 0.0.0.0 --port 8188 --disable-smart-memory")

In [ ]:
# Starting the Web UI with pinggy

%cd /kaggle/working/ComfyUI
!python /kaggle/working/pinggy.py --command='/kaggle/working/venv/bin/python /kaggle/working/ComfyUI/main.py' --port=8188

In [ ]:
# Install Zrok (only needs to run once)

!mkdir /kaggle/working/zrok
%cd /kaggle/working/zrok
!rm zrok*.gz
!wget https://github.com/openziti/zrok/releases/download/v1.0.4/zrok_1.0.4_linux_amd64.tar.gz
!tar -xvf ./zrok*.gz 
!chmod a+x /kaggle/working/zrok/zrok 

### Create a Zrok account
Enter your email address in the email variable

In [ ]:
email = '####@gmail.com' # replace with your email

# --------------

cmd = '/kaggle/working/zrok/zrok invite'
log = '/kaggle/working/zrok/log.txt'

!pip install pexpect
!touch $log

import pexpect
import time
child = pexpect.spawn('bash')
child.sendline(f'{cmd} | tee {log}')
child.expect('enter and confirm your email address...')
time.sleep(1); child.sendline(email); time.sleep(1); child.send(chr(9)); time.sleep(1)
child.sendline(email); time.sleep(1); child.send('\n'); time.sleep(1); child.send(chr(9))
time.sleep(1); child.send('\r\n'); time.sleep(2); child.close()
!cat $log
!rm $log

### Enable Zrok 
Paste your Zrok token in the token variable

In [ ]:
# Enable Zrok (needs to run once per instance)
# Paste your Zrok token in the token variable

token = ""
!chmod a+x /kaggle/working/zrok/zrok 
!/kaggle/working/zrok/zrok enable $token

### Start the WebUI with Zrok

In [ ]:
# Start the WebUI with Zrok
%cd /kaggle/working/ComfyUI
command = '/kaggle/working/venv/bin/python /kaggle/working/ComfyUI/main.py'
port = '8188'
# ------------------------

!chmod a+x /kaggle/working/zrok/zrok 
cmd = f'{command} & /kaggle/working/zrok/zrok share public http://localhost:{port} --headless'
get_ipython().system(cmd)

---
# Model Management

## Install a model

Copy the model URL to the model_url field. Make sure the model can be accessed publicly, without being signed into a website.

In [ ]:
#### Install a model in permanent storage
# Make sure Persistence is set to "Files only" or "Variables and Files"
model_url = 'https://civitai.com/api/download/models/782002'
model_name = 'JuggernautXL.safetensors'

%cd $checkpoints
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

In [ ]:
# Install a LoRA in permanent storage
model_url = 'https://civitai.com/api/download/models/137124?type=Model&format=SafeTensor'
model_name = 'DreamArt.safetensors'

%cd /kaggle/working/ComfyUI/models/loras
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

In [ ]:
# Install a model in temporary storage
#model_url = 'https://civitai.com/api/download/models/160191?type=Model&format=SafeTensor&size=full&fp=fp16'
#model_name = 'YamersRealism.safetensors'
model_url = 'https://civitai.com/api/download/models/456751'
model_name = 'HelloWorld-XL.safetensors' 

%cd $temp_models
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

## Download a model for a custom node

In [ ]:
model_folder = '/kaggle/working/ComfyUI/custom_nodes/my_node/models'
model_url = ''
model_name = 'model.safetensors'

%cd $model_folder
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

---
# File Browser

## Install FileBrowser

In [ ]:
%cd /kaggle
!wget https://github.com/filebrowser/filebrowser/releases/download/v2.27.0/linux-amd64-filebrowser.tar.gz
!tar xvfz linux-amd64-filebrowser.tar.gz
!chmod a+x /kaggle/filebrowser
!/kaggle/filebrowser config init 
!/kaggle/filebrowser config set --auth.method=noauth > /dev/null
!/kaggle/filebrowser config set --branding.theme=dark > /dev/null
!/kaggle/filebrowser users add admin admin 
!/kaggle/filebrowser config export "/kaggle/config.json"

## Run FileBrowser

In [ ]:
%cd /kaggle
!chmod a+x /kaggle/filebrowser

!python /kaggle/working/pinggy.py --command='/kaggle/filebrowser -c "/kaggle/working/config.json"' --port=8080

# 
# Delete a model

In [ ]:
# List permanent models
!ls -la $checkpoints

# Delete a model
model_to_delete = '/kaggle/working/ComfyUI/models/checkpoints/model.safetensors'
!rm $model_to_delete

In [ ]:
# Check the size of a model
!du -sh /kaggle/working/ComfyUI/models/loras/harrlogos.safetensors

# 
# Delete everything in the working folder

In [ ]:
# Delete the working folder
!rm -rf /kaggle/working/*